# Lab 4.2 &mdash; Your Own Traces, over MCP

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Tool Calling &amp; MCP**

### What you'll do
- Point an agent at the Langfuse project your Day 3 traces land in
- Weigh what a large tool set costs you, in tools and in tokens
- Scope a server you cannot configure &mdash; from the client side
- Ask questions about your own observability data in English

> **How this lab works &mdash; it is different from the others.** There is nothing to fill in
> and nothing to score. You run the cells in order and watch a real agent reach a real Jira
> over MCP. The participant notebook and the solution notebook are the same file, on purpose:
> the point is to *see the protocol work* before Module 4 asks you to build one. Read the
> output of every cell &mdash; that is the lab.

> **Nothing to fill in.** Lab 4.1 connected a server *we* run. This one connects a server
> *Langfuse* runs, holding *your* data &mdash; and that changes what you can control.

## The use case

On Day 3 you instrument agents with Langfuse and every run lands as a trace in a project. Then the
questions start: how many calls did that agent make, which one was slow, what did it actually send.

All of that is behind an API. Langfuse also publishes it as an **MCP server**, so an agent can
answer those questions for you instead of you learning the query language.

Everything you need is already in this sandbox &mdash; no signup, no new key.

| | Lab 4.1 &mdash; Jira | Lab 4.2 &mdash; Langfuse |
|---|---|---|
| who runs it | we do, in our cluster | Langfuse do, in the JP region |
| credential | a Basic header we issue | a Basic header from **your** project keys |
| whose data | a shared scratch board | **your own traces** |
| tool set | we chose five | they publish 85 |
| scoping | server-side, our flags | **client-side only** |

In [ ]:
# ------------------------------------------------------------ Preflight: run me first
import os, json, time, base64, textwrap, subprocess, re, socket, urllib.request, urllib.error

HOST   = os.environ.get("LANGFUSE_HOST", "")
PK     = os.environ.get("LANGFUSE_PUBLIC_KEY", "")
SK     = os.environ.get("LANGFUSE_SECRET_KEY", "")
ENVTAG = os.environ.get("LANGFUSE_TRACING_ENVIRONMENT", "")
LABDIR = os.path.expanduser("~/work/lfmcp")      # home, not /tmp: /tmp is wiped on restart

MCP_URL = HOST.rstrip("/") + "/api/public/mcp" if HOST else ""
AUTH    = base64.b64encode(f"{PK}:{SK}".encode()).decode() if (PK and SK) else ""

def ready() -> bool:
    return bool(HOST and PK and SK)

if ready():
    print("endpoint   :", MCP_URL)
    print("credential : Basic base64(public:secret) -- built from keys already in your environment")
    print("your traces:", ENVTAG or "(environment tag not set)")
    os.makedirs(LABDIR, exist_ok=True)
    print("lab folder :", LABDIR)
else:
    print("Not configured. This lab reads three variables the sandbox already sets:")
    for n in ("LANGFUSE_HOST", "LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY"):
        print(f"  {n:22} {'set' if os.environ.get(n) else 'MISSING'}")
    print("\nEvery cell below skips cleanly until they are set. Ask the trainer.")

## Step 1 &mdash; Connect

The same shape as Lab 4.1: `type: remote`, a URL, one header. Only the values differ &mdash; and
this time the header is built from *your* project keys, so the server will answer about *your*
project and nobody else's.

In [ ]:
CONFIG = {
    "$schema": "https://opencode.ai/config.json",
    "provider": {
        "litellm": {
            "npm": "@ai-sdk/openai-compatible",
            "name": "LiteLLM Gateway",
            "options": {"baseURL": "{env:LAB_LLM_BASE_URL}", "apiKey": "{env:LITELLM_API_KEY}"},
            "models": {"qwen36-35b-a3b-lab": {"name": "Qwen3.6 35B A3B (lab)"}},
        }
    },
    "mcp": {
        "langfuse": {
            "type": "remote",
            "url": MCP_URL,
            "enabled": True,
            # The key pair IS the project scope: Langfuse resolves which project to
            # answer for from these credentials. Nothing else in the config names it.
            "headers": {"Authorization": "Basic " + AUTH},
        }
    },
}

def write_config(cfg):
    path = os.path.join(LABDIR, "opencode.json")
    with open(path, "w") as fh:
        json.dump(cfg, fh, indent=2)
    return path

def oc(*args, timeout=180):
    p = subprocess.run(["opencode", *args], cwd=LABDIR, capture_output=True, text=True, timeout=timeout)
    return re.sub(r"\x1b\[[0-9;?]*[a-zA-Z]", "", (p.stdout or "") + (p.stderr or ""))

if ready():
    write_config(CONFIG)
    print(oc("mcp", "list"))
else:
    print("skipped - see the preflight cell")

## Step 2 &mdash; Discover, and weigh it

`initialize`, then `tools/list` &mdash; the same two calls as Lab 4.1. What comes back is not the
same size.

In [ ]:
def rpc(method, params=None, sid=None, timeout=120):
    body = json.dumps({"jsonrpc": "2.0", "id": 1, "method": method, "params": params or {}}).encode()
    req = urllib.request.Request(MCP_URL, data=body, method="POST")
    req.add_header("Content-Type", "application/json")
    req.add_header("Accept", "application/json, text/event-stream")
    req.add_header("Authorization", "Basic " + AUTH)
    if sid:
        req.add_header("Mcp-Session-Id", sid)
    with urllib.request.urlopen(req, timeout=timeout) as r:
        raw, sess = r.read().decode(), r.headers.get("mcp-session-id")
    for line in raw.splitlines():
        if line.startswith("data:"):
            raw = line[5:].strip()
            break
    return json.loads(raw).get("result", {}), sess


if ready():
    init, session = rpc("initialize", {
        "protocolVersion": "2025-06-18", "capabilities": {},
        "clientInfo": {"name": "lab-4-2", "version": "1.0"}})
    print("serverInfo:", init.get("serverInfo"))

    tools = rpc("tools/list", {}, sid=session)[0]["tools"]
    names = [t["name"] for t in tools]
    mutating = [n for n in names if n.startswith(("create", "update", "delete"))]
    schema_bytes = len(json.dumps(tools))

    print(f"\ntools published : {len(names)}")
    print(f"  of which write: {len(mutating)}  (create / update / delete)")
    print(f"  schema weight : {schema_bytes:,} characters, roughly {schema_bytes//4:,} tokens")
    print(f"                  ...re-sent to the model on EVERY turn, before your question")
    print("\na few you can read:", ", ".join(n for n in names if n.startswith(("list", "get")))[:150], "...")
    print("a few you cannot undo:", ", ".join(n for n in names if n.startswith("delete"))[:150], "...")
else:
    print("skipped - see the preflight cell")

Two things in that output are worth sitting with.

**The token weight.** Those schemas are context you have already spent before the model reads your
question. Discovery is not free, and it recurs on every turn of every run.

**The `delete` tools.** They act on the project your own traces live in. Nothing here is malicious
&mdash; Langfuse publish a complete API and that is reasonable of them. But your agent has been
handed the destructive half along with the useful half, and the only thing standing between a
loosely worded prompt and `deleteDashboard` is the model's judgement.

## Step 3 &mdash; Scope it, from the client

In Lab 4.1 we cut the tool set with `--enabled-tools`, because we ran that server. Here we cannot:
it is Langfuse's process, and their API is deliberately complete.

So the boundary moves into your own config. `opencode` decides per tool: `allow` silently,
`ask` prompts you, `deny` refuses. Wildcards work, and the specific name wins over the pattern.

In [ ]:
CONFIG["permission"] = {
    "langfuse_delete*": "deny",    # never, not even if asked nicely
    "langfuse_create*": "ask",     # a human decides, in the moment
    "langfuse_update*": "ask",
}

if ready():
    write_config(CONFIG)
    print(json.dumps(CONFIG["permission"], indent=2))
    print("\nThe server still publishes all", len(names), "tools and still lists them.")
    print("What changed is which ones THIS client is willing to call.")
    print("\nNote what this does NOT do: the same key pair still works against the")
    print("Langfuse REST API directly, where none of these rules apply. Client-side")
    print("scoping constrains your agent, not your credential.")
else:
    print("skipped - see the preflight cell")

## Step 4 &mdash; Ask about your own data

In a terminal (**File &rarr; New &rarr; Terminal**), as in Lab 4.1.

If you have not run any Day 3 labs yet the counts will be small or zero &mdash; that is fine and it
is still a real answer about a real project.

In [ ]:
TASK = ("Use the langfuse MCP tools. How many observations are in this project, and what are the "
        "three most recent ones? Answer briefly. Do not create, update or delete anything.")

if ready():
    print("Open File > New > Terminal, then paste:\n")
    print(f"cd {LABDIR} && \\\n  opencode run --model litellm/qwen36-35b-a3b-lab \\\n    \"{TASK}\"")
    print("\nThen try one it is not allowed to do:\n")
    print(f"cd {LABDIR} && \\\n  opencode run --model litellm/qwen36-35b-a3b-lab \\\n"
          f"    \"Use the langfuse MCP tools to delete every dashboard in this project.\"")
else:
    print("skipped - see the preflight cell")

Run that second command and read the answer carefully, because it is not what most people expect.

The agent does not say *"I am not allowed to do that."* It says the tool **does not exist**. `deny`
is not a policy the model is asked to respect &mdash; the tool never reaches it, so there is nothing
to refuse and nothing to argue with. Measured on this sandbox: with `langfuse_delete*` denied, the
agent looked, found the dashboard, and then reported that the server offers no way to delete one.

That is the same shape as Lab 4.1's scoped Jira server, arrived at from the other end. A capability
the model was never offered is stronger than a capability it was told not to use.

## What this lab changed

Two servers, two labs, one protocol &mdash; and almost everything else different:

| | 4.1 Jira | 4.2 Langfuse |
|---|---|---|
| tools | 5, because we chose 5 | 85, because they publish 85 |
| where scoping happens | the server, our flag | the client, our config |
| who the credential is | one shared service account | your project |
| blast radius | a scratch board | your own observability data |

**The lesson that survives both:** the tool list is a grant. In 4.1 you granted it by starting a
process with certain flags. Here you granted it by writing a URL and a header, and then took some
of it back in a `permission` block. Either way the decision was yours, it lived in a config file,
and nothing in the protocol made it for you.

**And the caveat that survives both:** scoping the *client* does not scope the *credential*. Those
keys still work against the Langfuse REST API, where your `deny` rules mean nothing. If that
matters, the answer is a narrower key &mdash; not a longer config.

## Your turn

- Change `langfuse_delete*` to `"ask"` and re-run the delete prompt. Notice where the decision lands.
- Ask it something that needs two tools &mdash; *"which observation was slowest, and what model did
  it use?"* &mdash; and watch it chain.
- Keep this config. On Day 3, when a trace looks wrong, ask this agent before opening the UI.